End-to-End FT Worfklows
-----------
In this tutorial, we describe the full end-to-end workflow to take a circuit written in any gate set to one written in a suitable gate set for many QECs (Quantum Error Correcting codes), the Clifford + T gate set.

This tutorial requires the `bqskit-ft` extension package and can be install via pip:
```bash
pip install bqskit-ft
```

Contents
-----------
* [Continuous Gate Set Re-Targeting](#continuous-gate-set-retargeting)
* [Decomposing to a Dicrete Gate Set](#fault-tolerant-gate-sets)
    * [Multi-Qubit Gate Synthesis] (#dyadic-phase-fixing)
    * [Single qubit U3 Gate Synthesis] (#algorithmic-error)
    * [ZXZXZ Decomposition with Gridsynth] (#gridsynth)
* [Algorithmic Error] (#algorithmic-error)
* [Compiling to Pauli Product Measurements] (#compiling-to-ppms)
* [Putting it all together] (#full-workflow)

Continuous Gate Set Re-Targeting
----------
One of the biggest advantages of the BQSKit workflow is our portability. Since we define our gates numerically, we can accept any gate type as long as it can be correctly represented by an underlying unitary. 

We can start by defining a circuit with various gates and unitaries. In our example we start by defining a circuit with random U3 gates, CNOT Gates, as well as some unitary operators corresponding to a 3-qubit QFT operator. Note that we can define any underlying unitary the same way.

The first step in our workflow decomposes our workflow to be more amenable to our final FT circuit. We'll define our worfklow to leverage multi-qubit synthesis to output a CNOT + continuous rotation circuit.

In [1]:
# Import the circuit primitives from basic BQSKit

In [2]:
# Define the Multi-qudit retargeting pass

Decomposing to a Discrete Gate Set
----------
One of the most significant challenges in Fault-Tolerant compilation is the switch from continous rotations (e.g $Rs$, $U3$ gates) to a set of discrete rotations that can be performed by the underlying error correction code.

By far the most common gate set studied in the literature is the Clifford + $T$ gate set. While exact implementations vary signficantly between architectures, the most significant cost comes from the realization of the $T$ gate. These operations are done via magic state injection, which requires the cultivation/distillation of a resource T state, that then gets injected into the full circuit.

BQSKit-FT uses 3 distinct techniques to minimize these expensive T gates:

1) Multi-Qubit Synthesis (Dynamic Phase Fixing)
2) Direct U3 Synthesis
3) ZXZXZ Decomposition with Gridsynth



Dyadic Phase Fixing
----------
We leverage the technique described in [2] to numerically minimize the number of $T$ gates that comes from continous angle decomposition. At a high level, we perform a greedy search to "fix" as many continuous $Rz$ angles to multiples of $\frac{\pi}{4}$ which can be implemented in a single $T$ gate.

As demonstrated in the paper, this technique can save a signficant percentage (up to 70%) of $T$ gates when compared to naive compilation. We discuss further in Section (something) how we can extend this technique when we target an architecture with many ancilla.


Direct U3 Synthesis
----------
After performing Dyadic Phase Fixing, we are left with our fixed $T$ gates, with the remaining set of continuous rotations.

To decompose this set of continuous rotations, we switch from numerical synthesis to an algebraic approach to gate decomposition. Following the technique described in [3], we can directly decompose *any* single qubit unitary to Clifford + T directly. While this is an expensive 8-dimensional lattce search, we see in practice that this search can be performed efficiently up to precisions of 10e-8, saving an additional 66% of $T$ gates when compared to gridsynth.

ZXZXZ Decomposition and GridSynth
----------
Finally, if there are remaining angles that need to be decomposed after the first 2 techniques, we default to the Ross-Selinger "gridsynth" algorithm described in [4]. This algorithm produces optimal length sequences of Clifford+$T$ gates approximating $R_Z(\theta)$ rotations. 

Any single-qubit unitary can be written as:
$$
    U = R_Z(\theta) \sqrt{X} R_Z(\phi) \sqrt{X} R_Z(\lambda)
$$
This means, on average, that this decomposition is 3x more expensive that direct $U3$ synthesis, but still represents the current state-of-the-art algorithm for high precision gate synthesis.

Algoritmic Error
-----------------

To this point, we have avoided talking about a fundamental component of fault-tolerant compilation: approximation error. Approximation error is necessary in fault-tolerant compilation as we try to express continous rotation gates from different algorithmic domains to a sequence of Clifford+$T$ gates.

We have set up our BQSKit workflows to bound the total approximation across the entire circuit, regardless of circuit size. We can do this via *circuit partitioning*

In [3]:
from bqskit.ir import Circuit
from bqskit.ir.gates import *

circuit = Circuit(2)
circuit.append_gate(HGate(), (0,))
circuit.append_gate(HGate(), (1,))
circuit.append_gate(CNOTGate(), (0,1))
circuit.append_gate(TGate(), (0,))
circuit.append_gate(HGate(), (0,))

3

In [4]:
from ppms.ppm_transpile import PPMTranspilePass


In [5]:
from bqskit.compiler import Compiler

compiler = Compiler(num_workers=1)
out_circ = compiler.compile(circuit, workflow=[PPMTranspilePass()])

Gate Set:  GateSet({HGate, CNOTGate, TGate})
Gate Table:  {HGate: 0, CNOTGate: 1, TGate: 2}
Serialized Gates:  [(False, b'\x80\x04\x95|\x01\x00\x00\x00\x00\x00\x00\x8c\x1abqskit.ir.gates.constant.h\x94\x8c\x05HGate\x94\x93\x94)\x81\x94}\x94(\x8c\r__cache_key__\x94h\x02)}\x94\x87\x94\x8c\x06_radix\x94K\x02\x8c\x05_utry\x94\x8c bqskit.qis.unitary.unitarymatrix\x94\x8c\rUnitaryMatrix\x94\x93\x94)\x81\x94}\x94(\x8c\x08_radixes\x94K\x02\x85\x94h\t\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\x02K\x02\x86\x94h\x14\x8c\x05dtype\x94\x93\x94\x8c\x03c16\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89C@\xcd;\x7ff\x9e\xa0\xe6?\x00\x00\x00\x00\x00\x00\x00\x00\xcd;\x7ff\x9e\xa0\xe6?\x00\x00\x00\x00\x00\x00\x00\x00\xcd;\x7ff\x9e\xa0\xe6?\x00\x00\x00\x00\x00\x00\x00\x00\xcd;\x7ff\x9e\xa0\xe6\xbf\x00\x00\x00\x00\x00\x00\x00\x00\x94t\x94b\x8c\x04_dim

In [6]:
for op in out_circ.operations():
    print(op.gate, op.location, op.params)

PPM ['X'] (0,) [0.25, 1.0]
PPM ['Z', 'Z'] (0, 1) [0.0, 1.0]
PPM ['X', 'X'] (0, 1) [0.0, 1.0]
